# 5교시 프로젝트 — 데이터로 파헤치는 히트곡의 비밀 🎧📊

> **사용법**
> - 🔵 **수업** = 강사와 함께 · 🟡 **셀프** = 집에서 직접 채우기(빈칸 `____`)
> - 정답본(`_solution`)에는 **실행 결과 + 💡해석**이 셀마다 있습니다.
> - 이 프로젝트의 목표는 '수업 중 완성'이 아니라 **집에서 한 번 더 직접 완주**하는 것!

---

## 🎬 오늘의 미션 — 음원 회사 데이터 분석가

여러분은 한 음악 스트리밍 회사의 데이터 분석가입니다. 기획팀장이 묻습니다.

> *"우리 다들 '신나고 빠른 곡이 인기 많다', '틱톡 때문에 곡이 짧아졌다',*
> *'K-pop은 템포가 빠르다' 고 믿잖아? 근데 **그거 진짜야?** 데이터로 확인해줘."*

오늘 우리는 **Spotify 글로벌 트랙 11.4만 곡**(114개 장르, BTS 포함)으로
이런 **'음악계 통념'을 데이터로 검증**합니다. 결과가 통념과 같을까요, 다를까요? 🤔

**오늘의 5단계**: 입수 → 정제(중복 제거) → 분포 보기 → 관계 분석 → K-pop/BTS 파헤치기 → 해석

## ① 입수 — 도구 + 한글 폰트 + 데이터  🔵

아래 셀로 도구·폰트·데이터를 준비합니다. Spotify 데이터는 **다운로드 없이** 깃허브에서 자동으로 받아옵니다(코랩 OK).

In [ ]:
# === 데이터 로드 설정 (로컬/코랩 자동) ===
import os, urllib.request
DATA_BASE = "https://raw.githubusercontent.com/acho98/gunyang-data/main/"
def data_path(fname):
    for _p in ("../data/" + fname, "data/" + fname):
        if os.path.exists(_p):
            return _p
    return DATA_BASE + fname

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

def set_korean_font():
    _cands = ["../data/BMDOHYEON.ttf", "data/BMDOHYEON.ttf", "BMDOHYEON.ttf"]
    if not any(os.path.exists(p) for p in _cands):
        try: urllib.request.urlretrieve(DATA_BASE + "BMDOHYEON.ttf", "BMDOHYEON.ttf")
        except Exception: pass
    for p in _cands:
        if os.path.exists(p):
            try:
                fm.fontManager.addfont(p)
                plt.rcParams["font.family"] = fm.FontProperties(fname=p).get_name()
                plt.rcParams["axes.unicode_minus"] = False
                return
            except Exception: pass
    print("(안내) 한글 폰트 미발견 — 영문 라벨은 정상.")
set_korean_font()

df = pd.read_csv(data_path("spotify_tracks.csv"))
print("입수 완료! 크기:", df.shape)

🤔 **확인**: 크기가 `(114000, 20)` 으로 나오면 성공입니다. 표가 잘 들어왔습니다.

### 첫인상 — `head()` 로 눈으로 보기  🔵

In [ ]:
df[["artists", "track_name", "track_genre", "popularity", "tempo", "energy", "danceability"]].head()

🤔 **직접 해석**: 어떤 컬럼들이 보이나요? 그중 '곡의 분위기/특성'을 나타내는 수치 변수를 2~3개 골라보세요.

## ② 정제 — 숨어 있는 중복 곡 제거  🔵

실제 데이터엔 항상 함정이 있습니다. 이 데이터는 **한 곡이 여러 장르에 중복** 수록돼 있어요
(예: 같은 곡이 'dance'와 'pop'에 동시에). 곡 단위로 분석하려면 **중복을 제거**해야 합니다.
먼저 중복이 얼마나 되는지 확인합니다.

In [ ]:
print("전체 행 수      :", len(df))
print("고유한 곡(track_id):", df["track_id"].nunique())
print("중복된 행 수     :", len(df) - df["track_id"].nunique())

🤔 **확인**: 전체 행 수와 고유 곡 수의 차이가 곧 중복 수입니다. 몇 개가 중복인가요?

### 중복 제거 — 곡 단위 표 만들기  🟡
`drop_duplicates("track_id")` 로 곡 하나당 한 줄만 남긴 `songs` 표를 만드세요.
> 💡 힌트: `df.drop_duplicates("track_id")`

In [ ]:
# TODO: track_id 기준 중복 제거 → songs
songs = df.drop_duplicates("____").copy()
print("곡 단위 표 크기:", songs.shape)

🤔 **직접 해석**: 중복 제거 후 몇 곡이 남았나요? 원본 `df`(11.4만)와 비교해보세요.

## ③ 분포 보기 — 히스토그램으로 '보통'을 파악  🔵

분석의 시작은 **"보통 어떤 값인가"** 입니다. 곡들의 **템포(BPM)** 가 어디에 몰려 있는지 히스토그램으로 봅니다.

In [ ]:
sns.histplot(data=songs, x="tempo", bins=40)
plt.title("곡 템포(BPM) 분포")
plt.xlabel("템포 (BPM)")
plt.show()
print("평균 템포:", round(songs["tempo"].mean(), 1), "BPM / 중앙값:", round(songs["tempo"].median(), 1))

🤔 **직접 해석**: 템포가 주로 어느 구간(BPM)에 몰려 있나요? 평균은 약 몇 BPM인가요?

### 직접 그려보기 — 댄서빌리티 분포  🟡
곡의 **댄서빌리티(`danceability`, 0~1 춤추기 좋은 정도)** 분포를 히스토그램으로 그려보세요.
> 💡 힌트: `sns.histplot(data=songs, x="danceability")`

In [ ]:
# TODO: 댄서빌리티(danceability) 분포를 히스토그램으로
sns.histplot(data=songs, x="____", bins=40)
plt.title("댄서빌리티 분포")
plt.show()
print("평균 댄서빌리티:", round(songs["danceability"].mean(), 3))

🤔 **직접 해석**: 댄서빌리티가 주로 어느 값에 몰려 있나요? 분포 모양이 한쪽으로 치우쳤나요, 가운데가 볼록한가요?

## ④ 관계 분석 (1) — "신나는 곡이 인기 많다"는 진짜일까?  🔵 (통념 검증 1)

팀장님의 첫 번째 통념: *"에너지 넘치고 춤추기 좋은 곡이 인기 많다."*
정말일까요? **댄서빌리티(x) ↔ 인기도(y)** 산점도로 확인합니다.

In [ ]:
sns.scatterplot(data=songs.sample(3000, random_state=1), x="danceability", y="popularity", alpha=0.3)
plt.title("댄서빌리티 vs 인기도 (표본 3000곡)")
plt.show()
print("상관계수(댄서빌리티-인기도):", round(songs["danceability"].corr(songs["popularity"]), 3))

🤔 **직접 해석**: 점들이 우상향/우하향 같은 방향을 보이나요, 아니면 구름처럼 흩어졌나요? 상관계수는 0에 가깝나요?

### 그럼 다른 수치들은? — 인기도와의 상관 한 번에  🟡
여러 음향 수치와 `popularity` 의 상관계수를 한 번에 봅니다. 빈칸에 `popularity` 를 넣어 완성하세요.
> 💡 힌트: `songs[cols].corr()["popularity"]`

In [ ]:
# TODO: 음향 수치들과 인기도(popularity)의 상관계수
cols = ["popularity", "danceability", "energy", "valence", "tempo", "loudness", "duration_ms"]
songs[cols].corr()["____"].round(3)

🤔 **직접 해석**: 가장 큰 상관계수도 얼마인가요(절댓값)? '음향 수치로 인기를 예측할 수 있다'고 말할 수 있을까요?

## ④ 관계 분석 (2) — "틱톡 때문에 짧은 곡이 인기"는 진짜일까?  🟡 (통념 검증 2)

두 번째 통념: *"요즘은 틱톡 영향으로 **짧은 곡**이 인기 많다."*
먼저 곡 길이를 **분(minute)** 으로 바꿔 분포를 보고, **짧은 곡 vs 긴 곡**의 평균 인기도를 비교합니다.
> 💡 힌트: 길이(분) = `duration_ms / 60000`

In [ ]:
# TODO: 곡 길이(분) = duration_ms / 60000, 그리고 짧은 곡/긴 곡 인기 비교
songs["분"] = songs["duration_ms"] / ____
print("평균 곡 길이:", round(songs["분"].mean(), 2), "분 / 중앙값:", round(songs["분"].median(), 2), "분")

short = songs[songs["분"] < 3]["popularity"].mean()
long_ = songs[songs["분"] > 4]["popularity"].mean()
print(f"3분 미만 곡 평균 인기도: {short:.1f}")
print(f"4분 초과 곡 평균 인기도: {long_:.1f}")

🤔 **직접 해석**: 짧은 곡과 긴 곡 중 평균 인기도가 높은 쪽은? '짧은 곡이 인기'라는 통념이 이 데이터에서 맞았나요?

> 📌 **솔직한 한계**: 이 데이터엔 **발매 연도**가 없어서 "해마다 곡이 짧아졌는지"의 *시간 변화*는 볼 수 없습니다.
> 그래서 우리는 "길이 ↔ 인기" 관계로 통념을 **간접 검증**했어요. *"어떤 데이터로 무엇까지 말할 수 있는가"* 를 아는 것도 분석가의 실력입니다.

## ④ 관계 분석 (3) — 장르마다 '음향 지문'이 있다  🔵

음향 수치가 인기는 못 가르지만, **장르의 색깔**은 또렷하게 보여줍니다.
대표 장르 5개의 **에너지(`energy`)** 분포를 **박스플롯**으로 비교합니다.

In [ ]:
genres = ["classical", "acoustic", "k-pop", "hip-hop", "edm"]
sub = df[df["track_genre"].isin(genres)]
sns.boxplot(data=sub, x="track_genre", y="energy", order=genres)
plt.title("장르별 에너지 분포")
plt.show()

🤔 **확인**: 에너지가 가장 낮은 장르와 높은 장르는? K-pop은 어디쯤 위치하나요?

### 직접 비교 — 장르별 템포  🟡
같은 5개 장르의 **템포(`tempo`)** 분포를 박스플롯으로 비교해보세요.
> 💡 힌트: `sns.boxplot(data=sub, x="track_genre", y="tempo", order=genres)`

In [ ]:
# TODO: 장르별 템포(tempo) 분포를 박스플롯으로
sns.boxplot(data=sub, x="track_genre", y="____", order=genres)
plt.title("장르별 템포(BPM) 분포")
plt.show()

🤔 **직접 해석**: 어느 장르의 템포가 가장 빠르고 느린가요? K-pop의 템포는 높은 편인가요, 중간인가요?

## ⑤ 하이라이트 — "K-pop은 템포가 빠르다"는 진짜일까? 🇰🇷  🔵 (통념 검증 3)

마지막 통념: *"K-pop은 신나고 빠른 곡이 많다."* 우리의 직감이죠.
**K-pop 장르의 평균 템포 vs 전체 평균 템포** 를 직접 비교합니다.

In [ ]:
kpop = df[df["track_genre"] == "k-pop"]
print(f"전체 평균 템포 : {df['tempo'].mean():.1f} BPM")
print(f"K-pop 평균 템포: {kpop['tempo'].mean():.1f} BPM")
print(f"K-pop 평균 인기도: {kpop['popularity'].mean():.1f}  (114개 장르 중 인기 2위!)")

🤔 **확인**: K-pop 평균 템포는 전체 평균보다 높나요, 낮나요? '빠르다'는 통념이 맞았나요?

### 직접 파헤치기 — BTS 곡들의 음향 프로필  🟡
이 데이터엔 **BTS 곡**도 있습니다! BTS 곡만 골라(중복 제거) 평균 음향 수치를 구해보세요.
> 💡 힌트: 문자열 포함 검색 `songs["artists"].str.contains("BTS")`

In [ ]:
# TODO: artists에 'BTS'가 포함된 곡만 골라 평균 음향 수치
bts = songs[songs["artists"].str.contains("____", case=False, na=False)]
print("데이터 속 BTS 곡 수:", len(bts))
print(bts[["tempo", "energy", "danceability", "popularity"]].mean().round(2))

🤔 **직접 해석**: BTS 곡은 몇 곡인가요? BTS의 평균 인기도(약 67)는 전체 곡 평균(약 33)과 비교해 어떤가요?

## ⑥ 해석 — 팀장님께 보고서 한 문단  🟡

오늘 검증한 통념 3개의 결과를 **한 번에 출력**하고, 맨 아래 `[보고]` 에 **내 결론을 한 문단** 적어보세요.

In [ ]:
# TODO: 빈칸을 채워 세 통념의 검증 결과를 출력하고, [보고]를 직접 한 문단 적어보세요
print("=== 음악계 통념, 데이터로 검증한 결과 ===")
print(f"통념1 '신나는 곡이 인기' → 댄서빌리티-인기 상관 {songs['danceability'].corr(songs['popularity']):.2f}")
print(f"통념2 '짧은 곡이 인기'   → 3분미만 {songs[songs['분']<3]['popularity'].mean():.1f} vs 4분초과 {songs[songs['분']>4]['popularity'].mean():.1f}")
print(f"통념3 'K-pop은 빠르다'   → K-pop {df[df.track_genre=='k-pop']['tempo'].mean():.0f} vs 전체 {df['tempo'].mean():.0f} BPM")
print()
print("[보고] (여기에 내가 알아낸 것을 한 문단으로 적어보세요)")

🤔 **직접 해석**: 세 통념의 검증 결과를 종합해 '무엇을 알아냈는가'를 한 문단으로. 가장 의외였던 결과 하나를 꼭 포함하세요.

## 🎉 수고하셨습니다 — '통념 파괴' 분석 완주!

오늘 여러분은 11.4만 곡으로 **세 가지 음악계 통념을 데이터로 검증**했고, **셋 다 틀렸다**는 걸 직접 밝혀냈습니다.

```
 ① 입수      ② 정제          ③ 분포          ④ 관계분석          ⑤ 해석
11.4만곡 →  중복 2.4만 제거 → 히스토그램 →  산점도·상관·박스플롯 →  "통념은 틀렸다"
```

**오늘의 가장 큰 교훈**:
> 우리가 '당연하다'고 믿던 것(신나는 곡=인기, 짧은 곡=인기, K-pop=빠름)이 **데이터 앞에서 줄줄이 무너졌습니다.**
> AI에게 "히트곡 공식 알려줘"라고 물으면 그럴듯한 답을 줄 거예요. 하지만 **"진짜야?" 하고 데이터로 따져보는 것** — 그게 분석가의 일입니다.

> ## "도구는 변해도, 좋은 질문을 던지고 결과를 해석하는 힘은 사람의 몫이다."

> 🏠 **셀프스터디**: 이 데이터로 또 다른 통념을 직접 검증해보세요.
> 예) "슬픈 곡(`valence` 낮음)이 더 인기 있을까?", "내가 좋아하는 아티스트의 음향 프로필은?" — 방법은 오늘과 똑같습니다! 🚀